In [ ]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium', 'figure.figsize': (18, 8),
          'axes.labelsize': 'medium', 'axes.titlesize': 'large',
          'xtick.labelsize': 'medium', 'ytick.labelsize': 'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import datetime
import warnings
warnings.filterwarnings('ignore')
from itertools import product
from dataclasses import replace

import pytz
NYC = pytz.timezone('America/New_York')

import sys
sys.path.append('../../')

# SFR Kink-Fading Grid Search

Systematic parameter sweep to find optimal configs for Sharpe, win rate, and P&L.

**Grid dimensions:** structure, z-score window, entry threshold, exit mode, regime filters, PCA residual

---
## 1. Load Data (once, reused across all backtests)

In [ ]:
from BT.signals.sfr_kink_fade import (
    KinkFadeConfig, KinkStructure, run_backtest,
    compute_pca_residual_rates, compute_strip_vol_proxy,
)
from BT.signals.sfr_cal_spread_rv import SFRCalSpreadRVConfig, load_rate_panel
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.TimeseriesBuilder import TimeseriesBuilder

DATA_START = '2024-01-01'
BT_START   = '2024-06-01'
BT_END     = 'live'

curve_mdp = IRSwapsMDP(source='BARCHART_STIRF-RL')
ts_builder = TimeseriesBuilder()

sfr_config = SFRCalSpreadRVConfig(
    n_contracts=12, zscore_window=60, vol_window=20,
    constant_maturity=True, source='BARCHART_STIRF-RL', curve='USD-SOFR-1D-Q12STIRT',
)

start = NYC.localize(datetime.datetime.fromisoformat(DATA_START).replace(hour=18))
print(f'Loading rate panel ({DATA_START} -> {BT_END})...')
rates = load_rate_panel(sfr_config, start=start, end=BT_END,
                        curve_mdp=curve_mdp, ts_builder=ts_builder)
print(f'  {rates.shape[0]} dates x {rates.shape[1]} contracts')

# Pre-compute PCA residual rates (expensive, do once)
print('Computing PCA residual rates (n_components=3, window=252)...')
pca_residual_rates = compute_pca_residual_rates(rates, n_components=3, window=252)
print('  Done.')

# Build backtest datetime grid
bt_start_dt = NYC.localize(datetime.datetime.fromisoformat(BT_START).replace(hour=17))
bt_end_dt = NYC.localize(datetime.datetime.now()) if BT_END == 'live' else \
            NYC.localize(datetime.datetime.fromisoformat(BT_END).replace(hour=17))
bt_dates = pd.bdate_range(bt_start_dt, bt_end_dt, tz=NYC)
bt_datetimes = [d.to_pydatetime() for d in bt_dates]
print(f'Backtest grid: {len(bt_datetimes)} steps ({bt_start_dt.date()} -> {bt_end_dt.date()})')

---
## 2. Define Parameter Grid

In [ ]:
# Base config (shared across all grid runs)
BASE = {
    'source': 'BARCHART_STIRF-RL',
    'curve': 'USD-SOFR-1D-Q12STIRT',
    'n_contracts': 12,
    'constant_maturity': True,
    'vol_window': 20,
    'carry_horizon': 1,
    'percentile_window': 60,
    'halflife_window': 120,
    'no_duplicate_structures': True,
    'belly_bpv': 100_000,
    'regime_max_adf_pvalue': None,   # disabled — not stationary over this sample
    'regime_min_halflife_days': 3.0,
    'regime_max_halflife_days': 120.0,
}

# Grid axes
GRID = {
    'structures': [
        ['bf_3m'],
        ['bf_6m'],
        ['df_3m'],
        ['bf_3m', 'df_3m'],
    ],
    'zscore_window': [20, 40, 60, 120],
    'entry_min_zscore': [1.0, 1.5, 2.0, 2.5],
    'exit_preset': [
        {'exit_mean_reversion': True,  'exit_stop_loss_sd': None,  'exit_take_profit_zscore': None,  'exit_max_holding_days': 44},
        {'exit_mean_reversion': True,  'exit_stop_loss_sd': 2.0,   'exit_take_profit_zscore': None,  'exit_max_holding_days': 22},
        {'exit_mean_reversion': False, 'exit_stop_loss_sd': None,  'exit_take_profit_zscore': None,  'exit_max_holding_days': 10},
        {'exit_mean_reversion': False, 'exit_stop_loss_sd': None,  'exit_take_profit_zscore': None,  'exit_max_holding_days': 21},
        {'exit_mean_reversion': False, 'exit_stop_loss_sd': 2.0,   'exit_take_profit_zscore': 0.5,   'exit_max_holding_days': 22},
    ],
    'regime_preset': [
        {'regime_fomc_blackout_days': None, 'regime_halflife_gated': False, 'regime_vol_filter': False, 'signal_mode': 'zscore'},
        {'regime_fomc_blackout_days': 5,    'regime_halflife_gated': False, 'regime_vol_filter': False, 'signal_mode': 'zscore'},
        {'regime_fomc_blackout_days': 5,    'regime_halflife_gated': True,  'regime_vol_filter': False, 'signal_mode': 'zscore'},
        {'regime_fomc_blackout_days': 5,    'regime_halflife_gated': False, 'regime_vol_filter': False, 'signal_mode': 'pca_residual'},
    ],
    'max_concurrent_trades': [3, 5],
}

# Build flat config list
configs = []
exit_labels = ['MR_44d', 'MR+S2_22d', 'Fix_10d', 'Fix_21d', 'TP05+S2_22d']
regime_labels = ['NoFilter', 'FOMC5', 'FOMC5+HL', 'FOMC5+PCA']

for structs in GRID['structures']:
    for zw in GRID['zscore_window']:
        for z_thresh in GRID['entry_min_zscore']:
            for ei, exit_preset in enumerate(GRID['exit_preset']):
                for ri, regime_preset in enumerate(GRID['regime_preset']):
                    for max_ct in GRID['max_concurrent_trades']:
                        cfg = dict(BASE)
                        cfg['structures'] = structs
                        cfg['zscore_window'] = zw
                        cfg['entry_min_zscore'] = z_thresh
                        cfg['max_concurrent_trades'] = max_ct
                        cfg.update(exit_preset)
                        cfg.update(regime_preset)

                        label = (f'{"+".join(structs)}|zw{zw}|z{z_thresh}|'
                                 f'{exit_labels[ei]}|{regime_labels[ri]}|ct{max_ct}')
                        cfg['_label'] = label
                        configs.append(cfg)

print(f'Total configs: {len(configs)}')
print(f'Grid: {len(GRID["structures"])} structs x {len(GRID["zscore_window"])} windows '
      f'x {len(GRID["entry_min_zscore"])} thresholds x {len(GRID["exit_preset"])} exits '
      f'x {len(GRID["regime_preset"])} regimes x {len(GRID["max_concurrent_trades"])} concurrency')

---
## 3. Run Grid Search

In [ ]:
import time

results = []
total = len(configs)
t0 = time.time()

for i, cfg in enumerate(configs):
    label = cfg.pop('_label')
    try:
        res = run_backtest(
            cfg, rates, curve_mdp, bt_datetimes,
            pca_residual_rates=pca_residual_rates,
        )
        res['label'] = label
        results.append(res)
    except Exception as e:
        results.append({'label': label, 'sharpe': np.nan, 'error': str(e)})

    if (i + 1) % 50 == 0 or i == total - 1:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (total - i - 1) / rate if rate > 0 else 0
        print(f'  [{i+1}/{total}] {elapsed:.0f}s elapsed, {eta:.0f}s ETA, {rate:.1f} runs/s')

elapsed = time.time() - t0
print(f'\nGrid search complete: {total} configs in {elapsed:.0f}s ({elapsed/60:.1f}m)')
print(f'Avg {elapsed/total:.1f}s per config')

---
## 4. Results Analysis

In [ ]:
df = pd.DataFrame(results)

# Parse label into columns
label_parts = df['label'].str.split('|', expand=True)
label_parts.columns = ['structure', 'zwindow', 'zthresh', 'exit', 'regime', 'concurrency']
df = pd.concat([df, label_parts], axis=1)

# Filter out errors and zero-trade configs
df_valid = df[df['n_trades'] > 0].copy()
print(f'Valid configs: {len(df_valid)} / {len(df)} ({len(df) - len(df_valid)} zero-trade)')
print(f'Sharpe range: [{df_valid["sharpe"].min():.2f}, {df_valid["sharpe"].max():.2f}]')
print(f'Win rate range: [{df_valid["win_rate"].min():.1%}, {df_valid["win_rate"].max():.1%}]')

In [ ]:
# Top 20 by Sharpe
print('=== TOP 20 BY SHARPE ===')
top_sharpe = df_valid.nlargest(20, 'sharpe')[
    ['label', 'sharpe', 'calmar', 'win_rate', 'hit_rate', 'final_mtm', 'max_dd',
     'n_trades', 'n_closed', 'avg_pnl', 'avg_hold_days']
].reset_index(drop=True)
display(top_sharpe)

In [ ]:
# Top 10 by Win Rate (min 10 closed trades)
print('=== TOP 10 BY WIN RATE (min 10 closed) ===')
df_enough = df_valid[df_valid['n_closed'] >= 10]
top_wr = df_enough.nlargest(10, 'win_rate')[
    ['label', 'win_rate', 'sharpe', 'n_closed', 'avg_pnl', 'final_mtm', 'max_dd']
].reset_index(drop=True)
display(top_wr)

# Top 10 by Final MTM
print('\n=== TOP 10 BY FINAL MTM ===')
top_pnl = df_valid.nlargest(10, 'final_mtm')[
    ['label', 'final_mtm', 'sharpe', 'win_rate', 'max_dd', 'n_trades', 'calmar']
].reset_index(drop=True)
display(top_pnl)

# Top 10 by Calmar
print('\n=== TOP 10 BY CALMAR ===')
top_calmar = df_enough.nlargest(10, 'calmar')[
    ['label', 'calmar', 'sharpe', 'win_rate', 'final_mtm', 'max_dd', 'n_trades']
].reset_index(drop=True)
display(top_calmar)

---
## 5. Marginal Effects (which parameter matters most?)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

dims = ['structure', 'zwindow', 'zthresh', 'exit', 'regime', 'concurrency']
for ax, dim in zip(axes.flat, dims):
    grp = df_valid.groupby(dim)['sharpe'].agg(['mean', 'median', 'std', 'count'])
    grp = grp.sort_values('median', ascending=False)

    x = range(len(grp))
    ax.bar(x, grp['median'], color='steelblue', alpha=0.7, label='Median')
    ax.errorbar(x, grp['median'], yerr=grp['std'], fmt='none', color='black', capsize=3)
    ax.set_xticks(x)
    ax.set_xticklabels(grp.index, rotation=45, ha='right', fontsize=7)
    ax.set_title(f'Sharpe by {dim}', fontweight='bold')
    ax.set_ylabel('Sharpe')
    ax.axhline(0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.show()

# Same for win rate
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for ax, dim in zip(axes.flat, dims):
    grp = df_enough.groupby(dim)['win_rate'].agg(['mean', 'median', 'std'])
    grp = grp.sort_values('median', ascending=False)

    x = range(len(grp))
    ax.bar(x, grp['median'], color='coral', alpha=0.7, label='Median')
    ax.errorbar(x, grp['median'], yerr=grp['std'], fmt='none', color='black', capsize=3)
    ax.set_xticks(x)
    ax.set_xticklabels(grp.index, rotation=45, ha='right', fontsize=7)
    ax.set_title(f'Win Rate by {dim}', fontweight='bold')
    ax.set_ylabel('Win Rate')
    ax.axhline(0.5, color='gray', linewidth=0.5)

plt.tight_layout()
plt.show()

---
## 6. Heatmaps (2D parameter interactions)

In [ ]:
import matplotlib.colors as mcolors

def plot_heatmap(df, row_dim, col_dim, metric, title, ax):
    pivot = df.groupby([row_dim, col_dim])[metric].median().unstack()
    cmap = 'RdYlGn'
    vmin, vmax = pivot.min().min(), pivot.max().max()
    if metric == 'sharpe':
        norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=max(vmax, 0.01))
    else:
        norm = None

    im = ax.imshow(pivot.values, cmap=cmap, aspect='auto', norm=norm)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_xlabel(col_dim)
    ax.set_ylabel(row_dim)
    ax.set_title(title, fontweight='bold')

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7,
                        color='white' if abs(val) > (vmax - vmin) * 0.4 else 'black')
    return im

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

plot_heatmap(df_valid, 'structure', 'zthresh', 'sharpe', 'Sharpe: Structure vs Z-Threshold', axes[0, 0])
plot_heatmap(df_valid, 'structure', 'zwindow', 'sharpe', 'Sharpe: Structure vs Z-Window', axes[0, 1])
plot_heatmap(df_valid, 'zthresh', 'exit', 'sharpe', 'Sharpe: Z-Threshold vs Exit', axes[1, 0])
plot_heatmap(df_valid, 'structure', 'regime', 'sharpe', 'Sharpe: Structure vs Regime', axes[1, 1])

plt.tight_layout()
plt.show()

---
## 7. Sharpe Distribution & Robustness

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Sharpe distribution
axes[0].hist(df_valid['sharpe'], bins=40, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='red', linewidth=1.5, linestyle='--')
axes[0].axvline(df_valid['sharpe'].median(), color='green', linewidth=1.5, label=f'Median={df_valid["sharpe"].median():.2f}')
axes[0].set_title('Sharpe Distribution (all configs)', fontweight='bold')
axes[0].set_xlabel('Sharpe')
axes[0].legend()

# Win rate distribution
axes[1].hist(df_enough['win_rate'], bins=30, color='coral', alpha=0.7, edgecolor='white')
axes[1].axvline(0.5, color='red', linewidth=1.5, linestyle='--')
axes[1].set_title('Win Rate Distribution (min 10 trades)', fontweight='bold')
axes[1].set_xlabel('Win Rate')

# Sharpe vs Max DD scatter
sc = axes[2].scatter(df_valid['max_dd'] / 1e6, df_valid['sharpe'],
                     c=df_valid['n_trades'], cmap='viridis', alpha=0.5, s=20)
axes[2].axhline(0, color='red', linewidth=0.5)
axes[2].set_title('Sharpe vs Max Drawdown', fontweight='bold')
axes[2].set_xlabel('Max Drawdown ($M)')
axes[2].set_ylabel('Sharpe')
plt.colorbar(sc, ax=axes[2], label='# Trades')

plt.tight_layout()
plt.show()

# Fraction of configs that are profitable
pct_positive_sharpe = (df_valid['sharpe'] > 0).mean()
pct_positive_pnl = (df_valid['final_mtm'] > 0).mean()
print(f'Configs with positive Sharpe: {pct_positive_sharpe:.1%}')
print(f'Configs with positive final MTM: {pct_positive_pnl:.1%}')
print(f'Median Sharpe: {df_valid["sharpe"].median():.3f}')
print(f'Median Win Rate: {df_enough["win_rate"].median():.1%}')

---
## 8. Best Config Deep Dive

In [ ]:
# Re-run the best Sharpe config with full tracking
best_idx = df_valid['sharpe'].idxmax()
best_cfg = df_valid.loc[best_idx, 'config']
print(f'Best Sharpe config: {df_valid.loc[best_idx, "label"]}')
print(f'Sharpe={df_valid.loc[best_idx, "sharpe"]:.3f}  '
      f'WinRate={df_valid.loc[best_idx, "win_rate"]:.1%}  '
      f'MTM=${df_valid.loc[best_idx, "final_mtm"]:,.0f}  '
      f'MaxDD=${df_valid.loc[best_idx, "max_dd"]:,.0f}')
print(f'\nConfig:')
for k, v in best_cfg.items():
    print(f'  {k}: {v}')

In [ ]:
# Re-run best config with full backtest object for P&L curve
from BT.signals.sfr_kink_fade_triggers import KinkFadeEntryTrigger, KinkFadeExitTrigger
from BT.signals.sfr_kink_fade import KinkFadeConfig, build_kink_fade_signal_table, compute_kink_curves, compute_rolling_halflife, compute_adf_snapshot, compute_pca_residual_rates
from BT.data_handler import TimeGrid
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy

best_config = KinkFadeConfig.from_dict(best_cfg)

if best_config.signal_mode == 'pca_residual':
    input_rates = pca_residual_rates
else:
    input_rates = rates

curves = compute_kink_curves(input_rates, best_config)
hl_data = {}
for ks, c in curves.items():
    if not c.empty:
        hl_data[ks] = compute_rolling_halflife(c, best_config.halflife_window)

st = build_kink_fade_signal_table(input_rates, best_config, halflife_data=hl_data)

entry_t = KinkFadeEntryTrigger(st, best_config)
exit_t = KinkFadeExitTrigger(st, best_config)
strat = QueryStrategy(name='best_config', triggers=[entry_t, exit_t], default_mdp=curve_mdp)

bt = QueryDrivenBacktest(time_grid=TimeGrid(bt_datetimes), strategy=strat, mdp=curve_mdp)
bt.run()

mtm = pd.Series(bt.mtm_history).sort_index()
daily_pnl = mtm.diff().dropna()
dd = mtm - mtm.cummax()

fig, axes = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={'height_ratios': [3, 1]})

axes[0].plot(mtm.index, mtm.values, linewidth=1.5, color='tab:cyan')
axes[0].axhline(0, color='black', linewidth=0.5)
std_d = daily_pnl.std()
sharpe = daily_pnl.mean() / std_d * np.sqrt(252) if std_d > 0 else 0
axes[0].set_title(f'Best Config: {df_valid.loc[best_idx, "label"]} | Sharpe={sharpe:.2f}', fontweight='bold')
axes[0].set_ylabel('P&L ($)')
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(dd.index, dd.values, 0, color='red', alpha=0.3)
axes[1].plot(dd.index, dd.values, color='red', linewidth=1)
axes[1].set_title(f'Drawdown | Max DD = ${dd.min():,.0f}', fontweight='bold')
axes[1].set_ylabel('Drawdown ($)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Closed position analysis
closed = bt.portfolio.closed_positions_log or []
if closed:
    reasons = {}
    for cp in closed:
        r = (cp.get('exit_meta') or {}).get('reason', 'unknown')
        reasons.setdefault(r, {'count': 0, 'pnl': 0})
        reasons[r]['count'] += 1
        reasons[r]['pnl'] += cp.get('realized_pnl', 0)

    print(f'\nClosed: {len(closed)} positions')
    for r, info in sorted(reasons.items(), key=lambda x: -x[1]['pnl']):
        print(f'  {r:20s}  N={info["count"]:>3d}  PnL=${info["pnl"]:>12,.0f}')

    # By structure
    struct_pnl = {}
    for cp in closed:
        st = (cp.get('position_meta') or {}).get('structure_type', '?')
        struct_pnl.setdefault(st, []).append(cp.get('realized_pnl', 0))
    print(f'\nBy structure:')
    for st, pnls in struct_pnl.items():
        wr = sum(1 for p in pnls if p > 0) / len(pnls)
        print(f'  {st:10s}  N={len(pnls):>3d}  Total=${sum(pnls):>12,.0f}  Win={wr:.1%}')

---
## 9. Export Results

In [ ]:
# Summary table for all valid configs
export_cols = ['label', 'sharpe', 'calmar', 'win_rate', 'hit_rate', 'final_mtm',
               'max_dd', 'n_trades', 'n_closed', 'avg_pnl', 'avg_hold_days',
               'structure', 'zwindow', 'zthresh', 'exit', 'regime', 'concurrency']
summary = df_valid[export_cols].sort_values('sharpe', ascending=False).reset_index(drop=True)

print(f'Summary: {len(summary)} valid configs')
print(f'\nTop 5 overall:')
display(summary.head())

print(f'\nBottom 5:')
display(summary.tail())

# Optionally save to CSV
# summary.to_csv('kink_fade_grid_results.csv', index=False)
# print('Saved to kink_fade_grid_results.csv')